# 1. Import & Load

In [1]:
!pip install pandas==1.3.5 numpy==1.26.4 --force-reinstall
# !pip install pandas==1.3.5 numpy==1.23.5 --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 40.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 35.4 MB/s eta 0:00:00
  Created wheel for pandas: filename=pandas-1.3.5-cp311-cp311-linux_x86_64.whl size=37464013 sha256=b41f9f59650d9aaab3a7097ad870adf6d4b1921ab6a0b77c1ac853c0e88474b6
  Stored in directory: /root/.cache/pip/wheels/8b/e7/6d/d4c288f419ab8fa07c1db6f606a2ae18ecf3dc2839d79a1c07
Successfully built pandas
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalling pytz-2025.2:
      Successfully uninstalled pytz-2025.2
  Attempting uninst

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
from datetime import datetime
import time
import re
import json
import gzip                                           ## UnicodeDecodeError 해결하기 위한 라이브러리
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')             ## numpy 버전이 1.26.4여야 glove embedding에서 gensim 불러올 때 에러가 없음..

pandas version: 1.3.5
numpy version: 1.26.4


In [3]:
# set data path
## 경로 설정하기
DATA_RAW_PATH = '/content/drive/MyDrive/박민경_논문1/dataset/rawdata'
DATA_PREP_PATH = '/content/drive/MyDrive/박민경_논문1/dataset/prep'
dataset_nm = 'Books'

In [ ]:
# Load data
## 데이터로드 함수 : 아마존 기준
def parse(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield json.loads(l)

def getDF(path):
  i = 0
  df = {}
  for d in parse(path):
    df[i] = d
    i += 1
  return pd.DataFrame.from_dict(df, orient='index')

## review data
df = getDF(f'{DATA_RAW_PATH}{dataset_nm}.jsonl.gz')
df_copy = df.copy()

print(df.shape)
df.head(5)

(29475453, 10)


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,1.0,Not a watercolor book! Seems like copies imo.,It is definitely not a watercolor book. The p...,[{'small_image_url': 'https://m.media-amazon.c...,B09BGPFTDB,B09BGPFTDB,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1642399598485,0,True
1,5.0,Updated: after 1st arrived damaged this one is...,Updated: after first book arrived very damaged...,[],0593235657,0593235657,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1640629604904,1,True
2,5.0,Excellent! I love it!,I bought it for the bag on the front so it pai...,[],1782490671,1782490671,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1640383495102,0,True
3,5.0,Updated after 1st arrived damaged. Excellent,Updated: after 1st arrived damaged the replace...,[],0593138228,0593138228,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1640364906602,0,False
4,5.0,Beautiful patterns!,I love this book! The patterns are lovely. I ...,[{'small_image_url': 'https://m.media-amazon.c...,0823098079,0823098079,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1637312253230,0,True


In [ ]:
# Load data - meta data
## 데이터로드 함수 : 아마존 기준
df_meta = getDF(f'{DATA_RAW_PATH}meta_{dataset_nm}.jsonl.gz')
df_meta_copy = df_meta.copy()

print(df_meta.shape)
df_meta.head(5)

(4448181, 16)


,main_category,title,subtitle,author,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Books,Chaucer,"Hardcover – Import, January 1, 2004",{'avatar': 'https://m.media-amazon.com/images/...,4.5,29,[],[],8.23,[{'large': 'https://m.media-amazon.com/images/...,[],Peter Ackroyd (Author),"[Books, Literature & Fiction, History & Critic...",{'Publisher': 'Chatto & Windus; First Edition ...,0701169850,None
1,Books,Notes from a Kidwatcher,First Edition,{'avatar': 'https://m.media-amazon.com/images/...,5.0,1,[Contains 23 selected articles by this influen...,"[About the Author, SANDRA WILDE, Ph.D., is wid...",3.52,[{'large': 'https://m.media-amazon.com/images/...,[],Sandra Wilde (Editor),"[Books, Reference, Words, Language & Grammar]",{'Publisher': 'Heinemann; First Edition (May 2...,0435088688,None
2,Books,Service: A Navy SEAL at War,"Hardcover – May 8, 2012",{'avatar': 'https://m.media-amazon.com/images/...,4.7,3421,"[Marcus Luttrell, author of the #1 bestseller,...","[Review, Praise for SERVICE""An action-packed.....",17.17,[{'large': 'https://m.media-amazon.com/images/...,[],"Marcus Luttrell (Author), James D. Hornfischer","[Books, Biographies & Memoirs, Leaders & Notab...","{'Publisher': 'Little, Brown and Company; 1st ...",0316185361,None
3,Books,Monstrous Stories #4: The Day the Mice Stood S...,"Paperback – October 29, 2013",None,4.4,40,"[Funny, light-hearted monster stories that are...",[],7.43,[{'large': 'https://m.media-amazon.com/images/...,[],Dr. Roach (Author),"[Books, Children's Books, Science Fiction & Fa...",{'Publisher': 'Scholastic Paperbacks; Reprint ...,0545425573,None
4,Buy a Kindle,Parker & Knight,Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.5,381,"[From REMINGTON KANE, the author of The Taken!...",[],0.0,[{'large': 'https://m.media-amazon.com/images/...,[],Remington Kane (Author) Format: Kindle Edition,"[Books, Mystery, Thriller & Suspense, Thriller...","{'Publication date': 'May 18, 2014', 'Language...",B00KFOP3RG,None


In [ ]:
# df.to_pickle(f'{DATA_RAW_PATH}/df_raw.pkl')
# df_meta.to_pickle(f'{DATA_RAW_PATH}/df_meta.pkl')

# 2. Prep

In [ ]:
# pkl data 불러오기
df = pd.read_pickle(f'{DATA_RAW_PATH}/df_raw.pkl')
df_meta = pd.read_pickle(f'{DATA_RAW_PATH}/df_meta.pkl')

### 1) merge with meta data
- category == 'Books'인 것만 필터링하기 위해

In [ ]:
# 1. merge review data with meta data
df_prep = pd.merge(
      df[['user_id', 'asin', 'text', 'rating', 'timestamp', 'parent_asin']],
      df_meta[['parent_asin', 'main_category']],
      on='parent_asin', how='left'
    )
df_prep = df_prep[df_prep['main_category'] == 'Books']          ## category => book인것만

print(df_prep.shape)
df_prep.head(5)

(24003884, 7)


,user_id,asin,text,rating,timestamp,parent_asin,main_category
0,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B09BGPFTDB,It is definitely not a watercolor book. The p...,1.0,1642399598485,B09BGPFTDB,Books
1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,0593235657,Updated: after first book arrived very damaged...,5.0,1640629604904,0593235657,Books
2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1782490671,I bought it for the bag on the front so it pai...,5.0,1640383495102,1782490671,Books
3,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,0593138228,Updated: after 1st arrived damaged the replace...,5.0,1640364906602,0593138228,Books
4,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,0823098079,I love this book! The patterns are lovely. I ...,5.0,1637312253230,0823098079,Books


### 2) rename columns

In [ ]:
# 2. rename columns
df_prep = df_prep.rename(columns = {'user_id':'userID', 'asin':'itemID', 'text':'reviewtext', 'timestamp':'review_date'})

print(df_prep.shape)
df_prep.head(5)

(24003884, 7)


,userID,itemID,reviewtext,rating,review_date,parent_asin,main_category
0,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B09BGPFTDB,It is definitely not a watercolor book. The p...,1.0,1642399598485,B09BGPFTDB,Books
1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,0593235657,Updated: after first book arrived very damaged...,5.0,1640629604904,0593235657,Books
2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1782490671,I bought it for the bag on the front so it pai...,5.0,1640383495102,1782490671,Books
3,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,0593138228,Updated: after 1st arrived damaged the replace...,5.0,1640364906602,0593138228,Books
4,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,0823098079,I love this book! The patterns are lovely. I ...,5.0,1637312253230,0823098079,Books


### 3) select data based on year, isna, duplicated, review length

In [ ]:
# 3. select data since 2019
df_prep['review_date'] = df_prep['review_date'].apply(lambda x: datetime.fromtimestamp(x/1000))     ### timestamp -> datetime
df_prep = df_prep[df_prep['review_date'].apply(lambda x: True if x.year >= 2019 else False)]  ### extract data from 2019

print(df_prep.shape)

(6458866, 7)


In [ ]:
# 4. drop duplicates
df_prep = df_prep.drop_duplicates()
print(df_prep.shape)

(6387417, 7)


In [ ]:
# 5. drop NaN
df_prep = df_prep.dropna()
print(df_prep.shape)

(6387417, 7)


In [ ]:
# 6. select data which len(review)>0
df_prep = df_prep[df_prep['reviewtext'].apply(lambda x: True if len(x)>0 else False)]
print(df_prep.shape)

(6387417, 7)


### 4) Text prep
- **주의사항!!!: 정규표현식으로 전처리할 때 무조건 '.' 포함시킬 것**
  
  - 이유: textrank에서 각 유저/아이템별로 주요 문장 추출할 때 구분점이 필요함!

In [ ]:
# 7. 텍스트 전처리
## 텍스트 전처리
def preprocess_text(text):
    # 영문자(a-zA-Z), 한글(가-힣), .만 남기고 나머지 문자 제거 후 소문자로 변환
    processed_text = re.sub(r'[^a-zA-Z가-힣\s.]', '', text).lower()

    return processed_text

## .으로만 이루어진 리뷰 데이터 제거
def is_meaningful_text(text):
    if not isinstance(text, str):
        return False
    cleaned = text.replace('.', '').strip()
    return len(cleaned) > 0


# df_prep_cp = df_prep.copy()
df_prep['reviewtext'] = df_prep['reviewtext'].apply(lambda x: preprocess_text(x))

# 적용
df_prep = df_prep[df_prep['reviewtext'].apply(is_meaningful_text)].copy()

print(df_prep.shape)
df_prep.head(5)

(6373675, 7)


,userID,itemID,reviewtext,rating,review_date,parent_asin,main_category
0,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B09BGPFTDB,it is definitely not a watercolor book. the p...,1.0,2022-01-17 06:06:38.485,B09BGPFTDB,Books
1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,0593235657,updated after first book arrived very damaged ...,5.0,2021-12-27 18:26:44.904,0593235657,Books
2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1782490671,i bought it for the bag on the front so it pai...,5.0,2021-12-24 22:04:55.102,1782490671,Books
3,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,0593138228,updated after st arrived damaged the replaceme...,5.0,2021-12-24 16:55:06.602,0593138228,Books
4,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,0823098079,i love this book the patterns are lovely. i a...,5.0,2021-11-19 08:57:33.230,0823098079,Books


In [ ]:
## 전처리 확인
df_prep[df_prep['reviewtext'].apply(lambda x: True if is_meaningful_text(x)==False else False)]

,userID,itemID,reviewtext,rating,review_date,parent_asin,main_category


### 5) Top-N core filtering

In [ ]:
# 7. 5-core 필터링
def filter_user_item(df, user_col='userID', item_col='itemID', min_review_nums = 5):
    while True:
      before_shape = df.shape[0]

      # 사용자당 아이템 수가 5개 이상
      user_item_counts = df.groupby(user_col)[item_col].nunique()
      df = df[df[user_col].isin(user_item_counts[user_item_counts >= min_review_nums].index)]

      # 아이템당 사용자 수가 5개 이상
      item_user_counts = df.groupby(item_col)[user_col].nunique()
      df = df[df[item_col].isin(item_user_counts[item_user_counts >= min_review_nums].index)]

      after_shape = df.shape[0]

      # 변화 없으면 종료
      if before_shape == after_shape:
          break

    return df

df_prep_5core = filter_user_item(df = df_prep, min_review_nums = 5)
print(df_prep_5core.shape)

(613727, 7)


In [ ]:
## 확인용: 사용자별 아이템 수(max, min, mean) -> 5-core
df_num_items_per_user = df_prep_5core.groupby('userID')['itemID'].nunique()
_mean_user_item = df_num_items_per_user.mean()
_max_user_item = df_num_items_per_user.max()
_min_user_item = df_num_items_per_user.min()

## 확인용: 아이템별 사용자 수(max, min, mean) -> 5-core
df_num_user_per_item = df_prep_5core.groupby('itemID')['userID'].nunique()
_mean_item_user = df_num_user_per_item.mean()
_max_item_user = df_num_user_per_item.max()
_min_item_user = df_num_user_per_item.min()

print(f'사용자별 아이템 수:, {round(_mean_user_item, 2)} (평균), {_max_user_item} (최대값), {_min_user_item} (최솟값)')
print(f'아이템별 사용자 수:, {round(_mean_item_user, 2)} (평균), {_max_item_user} (최대값), {_min_item_user} (최솟값)')

사용자별 아이템 수:, 12.14 (평균), 1131 (최대값), 5 (최솟값)
아이템별 사용자 수:, 13.17 (평균), 1256 (최대값), 5 (최솟값)


In [ ]:
# 8. 사용자 수, 아이템 수, 총 평점 수, 사용자별 아이템 수, 아이템 당 사용자 수, 희소(5-core)
rating_cnt = df_prep_5core['rating'].count()              ## 총 평점 수
user_cnt = df_prep_5core['userID'].nunique()              ## 사용자 수
item_cnt = df_prep_5core['itemID'].nunique()             ## 아이템 수
sparsity = 1 - (rating_cnt / (user_cnt*item_cnt))*100     ## sparsity

print(f'사용자 수: {user_cnt}')
print(f'아이템 수: {item_cnt}')
print(f'총 평점 수: {rating_cnt}')
print(f"Sparsity: {sparsity:.4f} ({sparsity * 100:.2f}%)")

사용자 수: 50554
아이템 수: 46612
총 평점 수: 613727
Sparsity: 0.9740 (97.40%)


### 6) save df as pkl

In [ ]:
# 10. pkl로 dataframe 저장
df_prep_5core.to_pickle(f'{DATA_PREP_PATH}book_data.pkl')          ## gensim vs numpy 간의 버전 이슈로 인해 numpy version: 1.26.4일 때 생성한 데이터 pkl로 저장함
df_prep = pd.read_pickle(f'{DATA_PREP_PATH}book_data.pkl')
print(df_prep.shape)

(613727, 7)


### 7) 사용자/아이템별로 리뷰 집합 생성
- **Textrank를 하려면 이 과정 패스하고 전단계까지 전처리해서 저장할 것!!!**

In [ ]:
# 사용자별 모으기
from bs4 import BeautifulSoup

grouped_user_reviews = df_prep[["userID", "reviewtext"]].groupby("userID")["reviewtext"].sum()
grouped_user_reviews = pd.DataFrame(grouped_user_reviews)
grouped_user_reviews.rename(columns = {"reviewtext":"userReviews"}, inplace = True)
grouped_user_reviews['userReviews'] = grouped_user_reviews['userReviews'].apply(
    lambda txt: BeautifulSoup(txt, "html.parser").get_text()
)
# grouped_user_reviews['userReviews'] = grouped_user_reviews['userReviews'].apply(
#     lambda txt: re.sub(r"\.", " ", txt).replace('\n', ' ')
# )

# 인덱스를 열로 변환
grouped_user_reviews.reset_index(inplace=True)

In [ ]:
book_user = pd.DataFrame({
    "userID": grouped_user_reviews["userID"].values,
    "userReviews": grouped_user_reviews["userReviews"].values})

print(book_user.shape)
book_user.head(3)

# 데이터프레임을 지정된 경로에 저장
book_user.to_pickle(f'{DATA_PREP_PATH}/book_user.pkl', protocol=4)

(50759, 2)


In [ ]:
book_user.head(3)

,userID,userReviews
0,AE226GBCLG76JH6DSG6E3UH362DQ,im so sad. i loved reading jack reacher. lee...
1,AE22EJZ4354VB7MN4IE2CDGHA2DQ,romantic comedy is a womens fiction novel abou...
2,AE22J4D76YM6ZBBRSUKCIK5BRJLQ,love the story. gave book to granddaughter bec...


In [ ]:
# 데이터프레임을 지정된 경로에서 불러오기
book_user = pd.read_pickle(f'{DATA_PREP_PATH}/book_user.pkl')
book_user.head()

,userID,userReviews
0,AE226GBCLG76JH6DSG6E3UH362DQ,im so sad. i loved reading jack reacher. lee...
1,AE22EJZ4354VB7MN4IE2CDGHA2DQ,romantic comedy is a womens fiction novel abou...
2,AE22J4D76YM6ZBBRSUKCIK5BRJLQ,love the story. gave book to granddaughter bec...
3,AE22JHFMLWWSBZZIJGRTPHB4S2IQ,love this book fir grandkidsgreat storyi love ...
4,AE22MFO6GCUJLXCWVQDLZDMWXQDA,i have never read a book that has touched me s...


In [ ]:
# 아이템별 모으기

grouped_item_reviews = df_prep[["itemID", "reviewtext"]].groupby("itemID")["reviewtext"].sum()
grouped_item_reviews = pd.DataFrame(grouped_item_reviews)
grouped_item_reviews.rename(columns = {"reviewtext":"itemReviews"}, inplace = True)
grouped_item_reviews['itemReviews'] = grouped_item_reviews['itemReviews'].apply(
    lambda txt: BeautifulSoup(txt, "html.parser").get_text()
)
# grouped_item_reviews['itemReviews'] = grouped_item_reviews['itemReviews'].apply(
#     lambda txt: re.sub(r"\.", " ", txt).replace('\n', ' ')
# )


# 인덱스를 열로 변환
grouped_item_reviews.reset_index(inplace=True)

In [ ]:
book_item = pd.DataFrame({
    "itemID": grouped_item_reviews["itemID"].values,
    "itemReviews": grouped_item_reviews["itemReviews"].values})

print(book_item.shape)

# 데이터프레임을 지정된 경로에 저장
book_item.to_pickle(f'{DATA_PREP_PATH}/book_item.pkl', protocol=4)

(46745, 2)


In [ ]:
book_item.head(3)

,itemID,itemReviews
0,0007149832,the book was in the condition as described. it...
1,0007450168,this is just such a lovely book i know my daug...
2,0007467850,kept me guessing and i never figurede it out. ...


In [ ]:
# 데이터프레임을 지정된 경로에서 불러오기
book_item = pd.read_pickle(f'{DATA_PREP_PATH}/book_item.pkl')
book_item.head()

,itemID,itemReviews
0,0007149832,the book was in the condition as described. it...
1,0007450168,this is just such a lovely book i know my daug...
2,0007467850,kept me guessing and i never figurede it out. ...
3,0007554850,this book serves as an important counterpart t...
4,0007922736,i had these when i was a child growing up and ...


# *리뷰 길이 확인


## Books

In [14]:
df_prep = pd.read_pickle(f'{DATA_PREP_PATH}/book_data.pkl')
df_prep.head(3)

,userID,itemID,reviewtext,rating,review_date,parent_asin,main_category
284,AFZUK3MTBIBEDQOPAK3OATUOUKLA,1646112814,i purchased this cookbook for ramen beginners ...,5.0,2022-12-24 00:36:43.582,1646112814,Books
288,AFZUK3MTBIBEDQOPAK3OATUOUKLA,1645174425,i purchased this cookbook for my niece who is ...,5.0,2022-08-10 02:13:59.384,1645174425,Books
289,AFZUK3MTBIBEDQOPAK3OATUOUKLA,0306923041,oh boy oh boy i have to admit i am not a vege...,5.0,2022-02-26 17:02:58.337,0306923041,Books


In [15]:
df_prep_cp = df_prep.copy()
df_prep_cp['review_sentence_len'] = df_prep_cp['reviewtext'].apply(lambda x: len(x.split('.')))
df_prep_cp['review_word_len'] = df_prep_cp['reviewtext'].apply(lambda x: len(x.split(' ')))
df_prep_cp['review_text_len'] = df_prep_cp['reviewtext'].apply(lambda x: len(x))

df_prep_cp.head(3)

,userID,itemID,reviewtext,rating,review_date,parent_asin,main_category,review_sentence_len,review_word_len,review_text_len
284,AFZUK3MTBIBEDQOPAK3OATUOUKLA,1646112814,i purchased this cookbook for ramen beginners ...,5.0,2022-12-24 00:36:43.582,1646112814,Books,13,217,1128
288,AFZUK3MTBIBEDQOPAK3OATUOUKLA,1645174425,i purchased this cookbook for my niece who is ...,5.0,2022-08-10 02:13:59.384,1645174425,Books,5,72,356
289,AFZUK3MTBIBEDQOPAK3OATUOUKLA,0306923041,oh boy oh boy i have to admit i am not a vege...,5.0,2022-02-26 17:02:58.337,0306923041,Books,17,244,1212


In [16]:
# 리뷰 평균 길이 확인
print(f'리뷰 평균 문장 길이: {round(df_prep_cp["review_sentence_len"].mean(), 2)}')
print(f'리뷰 평균 단어 길이: {round(df_prep_cp["review_word_len"].mean(), 2)}')
print(f'리뷰 평균 글자 길이: {round(df_prep_cp["review_text_len"].mean(), 2)}')

리뷰 평균 문장 길이: 7.94
리뷰 평균 단어 길이: 119.53
리뷰 평균 글자 길이: 647.84


## Movie

In [17]:
df_prep = pd.read_pickle(f'{DATA_PREP_PATH}/movie_data.pkl')
df_prep.head(3)

,rating,itemID,userID,reviewtext,date,year
0,1.0,B08X6JFYT9,AFGNIWCBLQT2QIXXKIW7Q6VREZRQ,so for all of those who think by buying the cu...,2021-03-01 15:57:45.651000064,2021
1,5.0,B07PND31HD,AFGNIWCBLQT2QIXXKIW7Q6VREZRQ,i wish there were more delightful movies like ...,2020-10-22 17:18:42.180999936,2020
2,1.0,B08DX1WFWK,AFGNIWCBLQT2QIXXKIW7Q6VREZRQ,spoilers this was just so ridiculous. so we ge...,2020-10-14 06:42:45.477999872,2020


In [18]:
df_prep_cp = df_prep.copy()
df_prep_cp['review_sentence_len'] = df_prep_cp['reviewtext'].apply(lambda x: len(x.split('.')))
df_prep_cp['review_word_len'] = df_prep_cp['reviewtext'].apply(lambda x: len(x.split(' ')))
df_prep_cp['review_text_len'] = df_prep_cp['reviewtext'].apply(lambda x: len(x))

df_prep_cp.head(3)

,rating,itemID,userID,reviewtext,date,year,review_sentence_len,review_word_len,review_text_len
0,1.0,B08X6JFYT9,AFGNIWCBLQT2QIXXKIW7Q6VREZRQ,so for all of those who think by buying the cu...,2021-03-01 15:57:45.651000064,2021,11,210,1100
1,5.0,B07PND31HD,AFGNIWCBLQT2QIXXKIW7Q6VREZRQ,i wish there were more delightful movies like ...,2020-10-22 17:18:42.180999936,2020,5,55,276
2,1.0,B08DX1WFWK,AFGNIWCBLQT2QIXXKIW7Q6VREZRQ,spoilers this was just so ridiculous. so we ge...,2020-10-14 06:42:45.477999872,2020,7,114,571


In [19]:
# 리뷰 평균 길이 확인
print(f'리뷰 평균 문장 길이: {round(df_prep_cp["review_sentence_len"].mean(), 2)}')
print(f'리뷰 평균 단어 길이: {round(df_prep_cp["review_word_len"].mean(), 2)}')
print(f'리뷰 평균 글자 길이: {round(df_prep_cp["review_text_len"].mean(), 2)}')

리뷰 평균 문장 길이: 4.25
리뷰 평균 단어 길이: 45.14
리뷰 평균 글자 길이: 242.18
